# Task 3: Data Cleaning

## Cleaning a Deliberately Messy Cafe Sales Dataset

**Objective:** Transform a messy transactional dataset into a clean, analysis-ready dataset using Python, pandas and numpy, while documenting every cleaning decision.


## 1. Import Libraries and Load the Raw Dataset

The original CSV is loaded without modifying it. Keeping the raw file unchanged allows us to compare the dataset before and after cleaning.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\dirty_cafe_sales.csv")
df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 2. Initial Data Quality Report

Before cleaning, we inspect:
- null values
- duplicate rows
- data types
- `ERROR` and `UNKNOWN` values
- numeric ranges
- unique values

This establishes a baseline for measuring the effect of cleaning.


In [2]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nNull values per column:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique values per column:")
print(df.nunique(dropna=False))

print("\n")
for col in df.columns:
    invalid_count = df[col].astype(str).str.upper().isin(["ERROR", "UNKNOWN"]).sum()
    print(f"{col}: {invalid_count} ERROR/UNKNOWN values")


Shape: (10000, 8)

Data types:
Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

Null values per column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate rows: 0

Unique values per column:
Transaction ID      10000
Item                   11
Quantity                8
Price Per Unit          9
Total Spent            20
Payment Method          6
Location                5
Transaction Date      368
dtype: int64


Transaction ID: 0 ERROR/UNKNOWN values
Item: 636 ERROR/UNKNOWN values
Quantity: 341 ERROR/UNKNOWN values
Price Per Unit: 354 ERROR/UNKNOWN values
Total Spent: 329 ERROR/UNKNOWN values
Payment Method: 599 ERROR/UNKNOWN values
Location: 696 ERROR/UNKNOWN val

## 3. Cleaning Decisions

### Invalid markers
`ERROR` and `UNKNOWN` are treated as missing values because they do not contain usable information.

### Missing categorical values
`Item`, `Payment Method`, and `Location` use **mode imputation**. These are categorical fields, and the most frequent valid category is a reasonable simple replacement for this practice dataset.

### Missing numeric values
`Quantity`, `Price Per Unit`, and `Total Spent` use **median imputation**. The median is less sensitive to unusually high or low transaction values than the mean.

### Missing transaction dates
Rows with missing or invalid dates are **removed** because a transaction date cannot be safely inferred from neighboring records: the data is not guaranteed to be sorted chronologically.

### Duplicates
Exact duplicate rows are removed. Duplicate count is checked before and after removal.

### Outliers
The IQR method is used. A statistical outlier is not automatically deleted. If an unusual value is still a plausible business transaction, it is retained.


In [3]:
# Treat ERROR and UNKNOWN as missing
invalid_markers = ["ERROR", "UNKNOWN"]

for col in df.columns:
    if col != "Transaction ID":
        df[col] = df[col].replace(invalid_markers, np.nan)

# Remove exact duplicates
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().copy()

print("Duplicates removed:", duplicate_count)


Duplicates removed: 0


## 4. Correct Numeric Data Types

The numeric columns are initially stored as text because of invalid strings. `pd.to_numeric(..., errors="coerce")` converts valid values to numbers and turns invalid entries into missing values.


In [4]:
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].dtypes


Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

## 5. Handle Missing Values

Categorical columns are filled with their mode. Numeric columns are filled with their median. Dates are converted to datetime and rows without a valid transaction date are removed.


In [5]:
# Categorical: mode
for col in ["Item", "Payment Method", "Location"]:
    mode_value = df[col].mode(dropna=True).iloc[0]
    print(f"{col} mode used for imputation: {mode_value}")
    df[col] = df[col].fillna(mode_value)

# Numeric: median
for col in numeric_cols:
    median_value = df[col].median()
    print(f"{col} median used for imputation: {median_value}")
    df[col] = df[col].fillna(median_value)

# Date: convert and remove rows where a valid date is unavailable
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
date_rows_removed = df["Transaction Date"].isna().sum()
df = df.dropna(subset=["Transaction Date"]).copy()

print("Rows removed because Transaction Date was missing/invalid:", date_rows_removed)


Item mode used for imputation: Juice
Payment Method mode used for imputation: Digital Wallet
Location mode used for imputation: Takeaway
Quantity median used for imputation: 3.0
Price Per Unit median used for imputation: 3.0
Total Spent median used for imputation: 8.0
Rows removed because Transaction Date was missing/invalid: 460


## 6. Standardise Data Types

IDs are stored as strings, monetary values as floats, quantity as integer, and transaction dates as datetime.


In [6]:
df["Transaction ID"] = df["Transaction ID"].astype("string")
df["Item"] = df["Item"].astype("string")
df["Quantity"] = df["Quantity"].round().astype("int64")
df["Price Per Unit"] = df["Price Per Unit"].astype("float64")
df["Total Spent"] = df["Total Spent"].astype("float64")
df["Payment Method"] = df["Payment Method"].astype("string")
df["Location"] = df["Location"].astype("string")
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

print(df.dtypes)


Transaction ID              string
Item                        string
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object


## 7. Outlier Detection Using the IQR Method

The IQR rule defines:
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Detected outliers are reviewed rather than automatically deleted. For this dataset, a high `Total Spent` value such as 25 can be a legitimate transaction (`5 × 5`), so it should be retained.


In [7]:
for col in numeric_cols:
    x = pd.to_numeric(
        pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\dirty_cafe_sales.csv")[col]
        .replace(["ERROR", "UNKNOWN"], np.nan),
        errors="coerce"
    ).dropna()

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = x[(x < lower) | (x > upper)]

    print(f"\n{col}")
    print("Q1:", q1, "Q3:", q3)
    print("IQR:", iqr)
    print("Lower bound:", lower)
    print("Upper bound:", upper)
    print("Outliers detected:", len(outliers))



Quantity
Q1: 2.0 Q3: 4.0
IQR: 2.0
Lower bound: -1.0
Upper bound: 7.0
Outliers detected: 0

Price Per Unit
Q1: 2.0 Q3: 4.0
IQR: 2.0
Lower bound: -1.0
Upper bound: 7.0
Outliers detected: 0

Total Spent
Q1: 4.0 Q3: 12.0
IQR: 8.0
Lower bound: -8.0
Upper bound: 24.0
Outliers detected: 259


## 8. Before vs After Summary

The final validation compares row count, duplicate count, null count and invalid marker count before and after cleaning.


In [8]:
raw = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\dirty_cafe_sales.csv")

before_invalid = sum(
    raw[col].astype(str).str.upper().isin(["ERROR", "UNKNOWN"]).sum()
    for col in raw.columns
)

after_invalid = sum(
    df[col].astype(str).str.upper().isin(["ERROR", "UNKNOWN"]).sum()
    for col in df.columns
)

summary = pd.DataFrame({
    "Metric": [
        "Row count",
        "Duplicate rows",
        "Null cells",
        "ERROR/UNKNOWN cells"
    ],
    "Before": [
        len(raw),
        raw.duplicated().sum(),
        raw.isna().sum().sum(),
        before_invalid
    ],
    "After": [
        len(df),
        df.duplicated().sum(),
        df.isna().sum().sum(),
        after_invalid
    ]
})

summary


,Metric,Before,After
0,Row count,10000,9540
1,Duplicate rows,0,0
2,Null cells,6826,0
3,ERROR/UNKNOWN cells,3256,0


## 9. Final Validation

The cleaned dataset should contain no missing values, no `ERROR`/`UNKNOWN` markers, no duplicate rows, and correct data types.


In [9]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nFinal shape:", df.shape)


Missing values:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Duplicate rows: 0

Data types:
Transaction ID              string
Item                        string
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object

Final shape: (9540, 8)


## 10. Save the Cleaned Dataset

The original raw dataset is preserved. The cleaned dataset is saved separately so the cleaning process remains reproducible.


In [10]:
output_path = r"C:\Users\HP\OneDrive\Documents\GitHub\OIBSIP\DataAnalytics-L1-Task3-CleaningData\cleaned_cafe_sales.csv"
df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully")


Cleaned dataset saved successfully
